In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [3]:

file_path = r"C:\Users\monay\OneDrive\Desktop\Final_Applied\feature_vectors_syscallsbinders_frequency_5_Cat (1).csv"

df = pd.read_csv(file_path, sep=',', on_bad_lines='skip')
print("Shape of dataset:", df.shape)
df.head()


Shape of dataset: (11598, 471)


,ACCESS_PERSONAL_INFO___,ALTER_PHONE_STATE___,ANTI_DEBUG_____,CREATE_FOLDER_____,CREATE_PROCESS`_____,CREATE_THREAD_____,DEVICE_ACCESS_____,EXECUTE_____,FS_ACCESS____,FS_ACCESS()____,...,utimes,vfork,vibrate,vibratePattern,wait4,watchRotation,windowGainedFocus,write,writev,Class
0,1,0,0,3,0,14,2,0,3,0,...,0,0,0,0,0,0,0,37,10,1
1,3,0,0,6,0,42,91,0,32,0,...,0,0,0,0,0,0,2,2838,46,1
2,2,0,0,4,0,23,3,0,17,2,...,0,0,0,0,0,0,1,111,20,1
3,1,0,0,4,0,27,9,0,36,0,...,0,0,0,0,0,0,7,987,197,1
4,3,0,0,11,0,18,3,0,16,0,...,0,0,0,0,0,0,1,98,25,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11598 entries, 0 to 11597
Columns: 471 entries, ACCESS_PERSONAL_INFO___ to Class
dtypes: int64(471)
memory usage: 41.7 MB


In [5]:
df.dtypes.unique()

array([dtype('int64')], dtype=object)

In [6]:
null = df.isnull().sum()
null_counts = null[null > 0]
null_counts

Series([], dtype: int64)

In [7]:
X = df.drop(columns=['Class'])
y = df['Class']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (11598, 470)
Target shape: (11598,)


In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [9]:
k_features = min(100, X_scaled.shape[1])

selector = SelectKBest(score_func=f_classif, k=k_features)
X_selected = selector.fit_transform(X_scaled, y)

selected_mask = selector.get_support()
selected_features = X.columns[selected_mask] # Corrected line

print("Number of selected features:", len(selected_features))
print("Selected features (first 20):", selected_features[:20])

X_final = X_selected

Number of selected features: 100
Selected features (first 20): Index(['CREATE_FOLDER_____', 'CREATE_PROCESS`_____', 'CREATE_THREAD_____',
       'EXECUTE_____', 'FS_ACCESS____', 'FS_ACCESS()____',
       'FS_ACCESS(CREATE)____', 'FS_ACCESS(CREATE__READ)__',
       'FS_ACCESS(CREATE__READ__WRITE)', 'FS_ACCESS(CREATE__WRITE)__',
       'FS_ACCESS(WRITE)____', 'FS_PIPE_ACCESS___', 'FS_PIPE_ACCESS(READ)___',
       'FS_PIPE_ACCESS(READ__WRITE)_', 'NETWORK_ACCESS(READ__WRITE)__',
       'NETWORK_ACCESS(WRITE__)__', 'TERMINATE_PROCESS', 'TERMINATE_THREAD',
       '__arm_nr_set_tls', 'access'],
      dtype='object')


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)


Train shape: (9278, 100) (9278,)
Test shape: (2320, 100) (2320,)


In [12]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
model_scores = model.predict(X_test)
accuracy = accuracy_score(y_test, model_scores)
precision = precision_score(y_test, model_scores, average='weighted', zero_division=0)
recall = recall_score(y_test, model_scores, average='weighted', zero_division=0)
f1 = f1_score(y_test, model_scores, average='weighted', zero_division=0)
print(f"Random Forest - Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")

Random Forest - Accuracy: 0.9397, Precision: 0.9409, Recall: 0.9397, F1-Score: 0.9397


In [16]:
model = SVC(random_state=42)
model.fit(X_train, y_train)
model_scores = model.predict(X_test)
accuracy = accuracy_score(y_test, model_scores)
precision = precision_score(y_test, model_scores, average='weighted', zero_division=0)
recall = recall_score(y_test, model_scores, average='weighted', zero_division=0)
f1 = f1_score(y_test, model_scores, average='weighted', zero_division=0)
print(f"SVC - Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")

SVC - Accuracy: 0.8073, Precision: 0.8094, Recall: 0.8073, F1-Score: 0.8052


In [18]:
# Encode labels to 0..n_classes-1 so XGBoost's sklearn wrapper accepts them
le = LabelEncoder()
le.fit(y)  
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
model.fit(X_train, y_train_enc)
model_scores = model.predict(X_test)

accuracy = accuracy_score(y_test_enc, model_scores)
precision = precision_score(y_test_enc, model_scores, average='weighted', zero_division=0)
recall = recall_score(y_test_enc, model_scores, average='weighted', zero_division=0)
f1 = f1_score(y_test_enc, model_scores, average='weighted', zero_division=0)
print(f"XGBoost - Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")

c:\Users\monay\OneDrive\Desktop\Final_Applied\nenv\Lib\site-packages\xgboost\training.py:199: UserWarning: [16:54:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost - Accuracy: 0.9422, Precision: 0.9427, Recall: 0.9422, F1-Score: 0.9422


In [19]:
model = KNeighborsClassifier()
model.fit(X_train, y_train)
model_scores = model.predict(X_test)
accuracy = accuracy_score(y_test, model_scores)
precision = precision_score(y_test, model_scores, average='weighted', zero_division=0)
recall = recall_score(y_test, model_scores, average='weighted', zero_division=0)
f1 = f1_score(y_test, model_scores, average='weighted', zero_division=0)
print(f"KNN - Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")

KNN - Accuracy: 0.8978, Precision: 0.9010, Recall: 0.8978, F1-Score: 0.8975


In [20]:
model = GaussianNB()
model.fit(X_train, y_train)
model_scores = model.predict(X_test)    
accuracy = accuracy_score(y_test, model_scores)
precision = precision_score(y_test, model_scores, average='weighted', zero_division=0)
recall = recall_score(y_test, model_scores, average='weighted', zero_division=0)
f1 = f1_score(y_test, model_scores, average='weighted', zero_division=0)
print(f"GaussianNB - Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")

GaussianNB - Accuracy: 0.5797, Precision: 0.6952, Recall: 0.5797, F1-Score: 0.5340
